# Ubunye Engine — End-to-End ML Pipeline
## Use Case: Titanic Survival Prediction

---

### Problem Statement

The RMS Titanic sank in 1912 after colliding with an iceberg. Of the 2,224 passengers and
crew, more than 1,500 died. **Can we predict which passengers were likely to survive based
on their age, class, sex, and fare paid?**

This notebook demonstrates the **complete ML lifecycle** using Ubunye Engine — from raw
data ingestion through to a versioned, production-ready model:

| Stage | What we cover |
|---|---|
| 1. Setup | Install, scaffold use case, create configs |
| 2. Config | Jinja2 templating, profiles (dev/prod), validation |
| 3. Ingestion | Task interface, lineage recording |
| 4. Feature Engineering | Pandas transforms, MLflow telemetry |
| 5. Model Training | `UbunyeModel`, `ModelTransform`, `PromotionGate` |
| 6. Registry | Register, promote dev → staging → production |
| 7. Prediction | Load from registry by stage, serve predictions |
| 8. Lineage | Inspect full audit trail via CLI |
| 9. Maintenance | Train v2, compare, rollback, archive |

> **Dataset:** Kaggle Titanic (`/kaggle/input/titanic/train.csv`, `test.csv`)
>
> **Note:** This notebook runs entirely without Apache Spark or cloud infrastructure.
> It exercises all engine contracts (Task, UbunyeModel, Registry, Lineage, Telemetry)
> using pandas DataFrames and local filesystem storage.

---
## Section 0 — Install & Setup

Install `ubunye-engine` directly from GitHub and set up MLflow local tracking.
All engine artifacts (lineage records, model store, MLflow runs) are written under
`/kaggle/working/pipelines/.ubunye/` so everything is co-located with the pipeline code.

In [ ]:
# Install ubunye-engine from GitHub (not yet on PyPI)
# mlflow is the optional telemetry backend used later
!pip install "git+https://github.com/ubunye-ai-ecosystems/ubunye_engine.git" mlflow -q

In [ ]:
import os, sys, json, uuid, time, pickle, pathlib, subprocess, textwrap
import pandas as pd
from dataclasses import dataclass
from typing import Any, Dict, Optional

# ── Directory layout ──────────────────────────────────────────────────────
# All pipeline code and artifacts live under PIPELINE_ROOT.
PIPELINE_ROOT = pathlib.Path("/kaggle/working/pipelines")
UBUNYE_DIR    = PIPELINE_ROOT / ".ubunye"
MODEL_STORE   = str(UBUNYE_DIR / "model_store")
LINEAGE_DIR   = str(UBUNYE_DIR / "lineage")
MLFLOW_DIR    = str(UBUNYE_DIR / "mlflow")

for d in [PIPELINE_ROOT, UBUNYE_DIR / "model_store",
          UBUNYE_DIR / "lineage", UBUNYE_DIR / "mlflow"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# ── MLflow local tracking (no server needed) ──────────────────────────────
import mlflow
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")

# ── Helper: run a CLI command and print output ────────────────────────────
def cli(*args):
    """Run `ubunye <args>` and print stdout/stderr."""
    result = subprocess.run(["ubunye"] + list(args), capture_output=True, text=True)
    output = result.stdout or result.stderr
    print("$", "ubunye", " ".join(str(a) for a in args))
    print(output)
    return result

# ── MockContext: stand-in for EngineContext when calling monitors directly ─
@dataclass
class MockContext:
    run_id:    str
    task_name: str
    profile:   str = "dev"

print("Setup complete.")
print(f"Pipeline root : {PIPELINE_ROOT}")
print(f"Model store   : {MODEL_STORE}")
print(f"Lineage dir   : {LINEAGE_DIR}")
print(f"MLflow dir    : {MLFLOW_DIR}")

---
## Section 1 — Scaffold Use Case Structure

**Problem it solves:** Starting a new project from scratch means creating the right
folder structure with boilerplate config and transformation files. `ubunye init` does
this in one command — you get a `config.yaml` and a `transformations.py` stub ready to edit.

Our pipeline has three stages, each as a separate task:

```
pipelines/
  titanic/
    ingestion/
      raw_ingest/          ← Stage 1: clean raw passenger data
    features/
      feat_engineering/    ← Stage 2: engineer predictive features
    training/
      survival_model/      ← Stage 3: train and register the model
```

In [ ]:
# ubunye init creates the folder skeleton with a config.yaml stub
# and a transformations.py Task subclass placeholder.
for package, task in [
    ("ingestion", "raw_ingest"),
    ("features",  "feat_engineering"),
    ("training",  "survival_model"),
]:
    cli("init",
        "-d", str(PIPELINE_ROOT),
        "-u", "titanic",
        "-p", package,
        "-t", task)

print("\nScaffolded task directories:")
for p in sorted(PIPELINE_ROOT.rglob("config.yaml")):
    print(" ", p.relative_to(PIPELINE_ROOT))

---
## Section 2 — Configs with Jinja2 Templating & Profiles

**Problem it solves:**  
In production pipelines, configs change between environments (dev vs prod) and daily runs
use different data partitions. Hard-coding these values leads to errors and environment drift.

Ubunye Engine renders `{{ variables }}` in YAML **before** Pydantic validation so you can:
- Use `{{ dt }}` for the daily data partition date (passed at runtime)
- Use `{{ env.DB_NAME }}` to read from environment variables
- Use `{{ env.S3_BUCKET | default('my-bucket') }}` with safe fallbacks
- Define `ENGINE.profiles.dev` / `ENGINE.profiles.prod` for per-environment Spark tuning

In [ ]:
# ── Stage 1: Ingestion config ─────────────────────────────────────────────
# Uses {{ dt }} for the daily partition and {{ env.HIVE_DB | default(...) }}
# for environment-specific database names.
# In production this config runs against a real Hive metastore;
# in this notebook we bypass the reader and feed pandas DataFrames directly.

ingest_dir = PIPELINE_ROOT / "titanic" / "ingestion" / "raw_ingest"

(ingest_dir / "config.yaml").write_text(textwrap.dedent("""\
    MODEL: "etl"
    VERSION: "1.0.0"

    ENGINE:
      spark_conf:
        spark.sql.shuffle.partitions: "50"
        spark.executor.memory: "4g"
      profiles:
        dev:
          spark.sql.shuffle.partitions: "4"
          spark.executor.memory: "512m"
          spark.master: "local[2]"
        prod:
          spark.sql.shuffle.partitions: "200"
          spark.executor.memory: "16g"

    CONFIG:
      inputs:
        raw_passengers:
          format: hive
          db_name: "{{ env.HIVE_DB | default('default') }}"
          tbl_name: "titanic_raw_{{ dt | default('19700101') }}"

      transform:
        type: noop

      outputs:
        clean_passengers:
          format: s3
          path: "s3a://{{ env.S3_BUCKET | default('my-bucket') }}/titanic/ingestion/{{ dt | default('19700101') }}"
          mode: overwrite
"""))

print("Ingestion config written.")

In [ ]:
# ── Stage 2: Feature engineering config ──────────────────────────────────
# Uses {{ mode }} to switch between dev (small shuffle) and prod (large shuffle).

feat_dir = PIPELINE_ROOT / "titanic" / "features" / "feat_engineering"

(feat_dir / "config.yaml").write_text(textwrap.dedent("""\
    MODEL: "etl"
    VERSION: "1.0.0"

    ENGINE:
      spark_conf:
        spark.sql.shuffle.partitions: "50"
      profiles:
        dev:
          spark.sql.shuffle.partitions: "4"
          spark.master: "local[2]"
        prod:
          spark.sql.shuffle.partitions: "400"

    CONFIG:
      inputs:
        clean_passengers:
          format: s3
          path: "s3a://{{ env.S3_BUCKET | default('my-bucket') }}/titanic/ingestion/{{ dt | default('19700101') }}"
          mode: overwrite

      transform:
        type: noop

      outputs:
        feature_set:
          format: s3
          path: "s3a://{{ env.S3_BUCKET | default('my-bucket') }}/titanic/features/{{ dt | default('19700101') }}"
          mode: overwrite

      monitors:
        - type: mlflow
          params:
            experiment: titanic_feature_engineering
            run_name:   feat_engineering_run
            tags:
              use_case: titanic
              stage: feature_engineering
"""))

print("Feature engineering config written.")

In [ ]:
# ── Stage 3: Model training config ───────────────────────────────────────
# Uses transform.type: model — the ModelTransform plugin.
# The registry block wires training directly into the model registry.
# promotion_gates enforce minimum quality before staging promotion.

train_dir = PIPELINE_ROOT / "titanic" / "training" / "survival_model"

(train_dir / "config.yaml").write_text(textwrap.dedent(f"""\
    MODEL: "ml"
    VERSION: "1.0.0"

    ENGINE:
      spark_conf:
        spark.sql.shuffle.partitions: "50"
      profiles:
        dev:
          spark.sql.shuffle.partitions: "4"
          spark.master: "local[2]"

    CONFIG:
      inputs:
        feature_set:
          format: s3
          path: "s3a://my-bucket/titanic/features/{{{{ dt | default('19700101') }}}}"
          mode: overwrite

      transform:
        type: model
        params:
          action: train
          model_class: "model.TitanicSurvivalModel"
          model_dir: "{train_dir}"
          registry:
            store: "{MODEL_STORE}"
            use_case: titanic
            auto_version: true
            promote_to: staging
            promotion_gates:
              min_accuracy: 0.72
              min_f1: 0.68

      outputs:
        model_metrics:
          format: s3
          path: "s3a://my-bucket/titanic/metrics"
          mode: overwrite
"""))

print("Training config written.")

---
## Section 3 — Validate & Plan Configs

**Problem it solves:**  
Config errors (wrong format name, missing required field, invalid Jinja template) only
surface at runtime — after Spark has started and readers have connected. That can mean
wasting 5–15 minutes before hitting a typo.

`ubunye validate` runs full Pydantic schema validation and Jinja2 template resolution
**before** Spark starts so you catch errors in seconds.  
`ubunye plan` prints the resolved I/O graph so you can confirm the data flow looks correct.

In [ ]:
# Validate all three task configs in one command.
# If any config has a schema error or unresolvable Jinja template, this exits non-zero.
for package, task in [
    ("ingestion", "raw_ingest"),
    ("features",  "feat_engineering"),
    ("training",  "survival_model"),
]:
    cli("validate",
        "-d", str(PIPELINE_ROOT),
        "-u", "titanic",
        "-p", package,
        "-t", task)

In [ ]:
# ubunye plan shows the resolved inputs → transform → outputs for each task.
# Useful to confirm Jinja variables resolved correctly before a production run.
for package, task in [
    ("ingestion", "raw_ingest"),
    ("features",  "feat_engineering"),
    ("training",  "survival_model"),
]:
    cli("plan",
        "-d", str(PIPELINE_ROOT),
        "-u", "titanic",
        "-p", package,
        "-t", task,
        "-dt", "20250101",   # data timestamp variable
        "-m", "DEV")

In [ ]:
# Demonstrate config profile merging: dev vs prod Spark settings.
# In production the engine calls merged_spark_conf(profile) before starting Spark.
from ubunye.config import load_config

cfg = load_config(str(ingest_dir))

dev_conf  = cfg.merged_spark_conf("dev")
prod_conf = cfg.merged_spark_conf("prod")

print("Profile comparison — spark.sql.shuffle.partitions:")
print(f"  dev  → {dev_conf.get('spark.sql.shuffle.partitions')}")
print(f"  prod → {prod_conf.get('spark.sql.shuffle.partitions')}")
print()
print("Profile comparison — spark.executor.memory:")
print(f"  dev  → {dev_conf.get('spark.executor.memory')}")
print(f"  prod → {prod_conf.get('spark.executor.memory')}")

---
## Section 4 — Data Ingestion

**Problem it solves:**  
Raw data from operational systems is messy — duplicate rows, wrong types, nulls in key
columns. The ingestion stage is the contract layer: it cleans and standardises raw data
before any downstream processing.

The `Task` interface is the user contract for transformations. You subclass it, write pure
data transformation logic in `transform()`, and the engine handles all I/O, monitoring,
and lineage recording around it.

**What we demonstrate here:**
- Implementing a `Task` subclass for ingestion
- Attaching `LineageRecorder` to capture a full provenance record
- Attaching `MLflowMonitor` to log run metadata to the experiment tracker

In [ ]:
from ubunye.core.interfaces import Task

class RawIngestTask(Task):
    """
    Stage 1 — Ingestion.

    Reads raw Titanic passenger data and applies minimal cleaning:
    - Drop rows with no survival label
    - Normalise column names to lowercase
    - Cast key columns to correct types
    - Remove obvious duplicates

    Output 'clean_passengers' is the bronze-layer dataset used downstream.
    """

    def setup(self) -> None:
        # self.config holds the full resolved config dict if needed
        self.required_cols = ["Survived", "Pclass", "Sex", "Age", "Fare"]

    def transform(self, sources: Dict[str, Any]) -> Dict[str, Any]:
        df: pd.DataFrame = sources["raw_passengers"]

        rows_in = len(df)

        # Normalise column names
        df.columns = df.columns.str.strip()

        # Drop rows with no survival label — unusable for training
        df = df.dropna(subset=["Survived"])

        # Cast types
        df["Survived"] = df["Survived"].astype(int)
        df["Pclass"]   = df["Pclass"].astype(int)
        df["Fare"]     = df["Fare"].astype(float)

        # Drop true duplicates (same PassengerId)
        df = df.drop_duplicates(subset=["PassengerId"])

        rows_out = len(df)
        print(f"  Ingestion | rows in: {rows_in}  rows out: {rows_out}  "
              f"dropped: {rows_in - rows_out}")
        print(f"  Null counts:\n{df[self.required_cols].isnull().sum().to_string()}")

        return {"clean_passengers": df}

print("RawIngestTask defined.")

In [ ]:
from ubunye.lineage.recorder import LineageRecorder
from ubunye.telemetry.mlflow import MLflowMonitor
from ubunye.telemetry.monitors import safe_call

# Load and validate the config
ingest_cfg = load_config(str(ingest_dir))

# Set up monitors: lineage + MLflow
lineage_recorder = LineageRecorder(store="filesystem", base_dir=LINEAGE_DIR)
mlflow_monitor   = MLflowMonitor(
    experiment="titanic_ingestion",
    run_name="raw_ingest_v1",
    tags={"use_case": "titanic", "stage": "ingestion"},
)
monitors = [lineage_recorder, mlflow_monitor]

# Build context (the engine creates this; we create it manually here)
ingest_ctx = MockContext(
    run_id=str(uuid.uuid4()),
    task_name="titanic/ingestion/raw_ingest",
)
ingest_config_dict = ingest_cfg.model_dump(mode="json")

# Signal task start to all monitors
for m in monitors:
    safe_call(m, "task_start", context=ingest_ctx, config=ingest_config_dict)

# Load raw data (in production the HiveReader does this)
raw_df = pd.read_csv("/kaggle/input/titanic/train.csv")

# Run the task
task = RawIngestTask(config=ingest_config_dict)
task.setup()

t0 = time.perf_counter()
ingest_outputs = task.transform({"raw_passengers": raw_df})
duration = time.perf_counter() - t0

clean_df = ingest_outputs["clean_passengers"]

# Signal task end to all monitors (lineage writes JSON; MLflow logs metrics)
for m in monitors:
    safe_call(m, "task_end",
              context=ingest_ctx,
              config=ingest_config_dict,
              outputs=ingest_outputs,
              status="success",
              duration_sec=duration)

print(f"\nIngestion complete in {duration:.3f}s  |  run_id: {ingest_ctx.run_id[:8]}...")
print(f"Clean dataset shape: {clean_df.shape}")
clean_df.head(3)

---
## Section 5 — Feature Engineering

**Problem it solves:**  
Raw passenger attributes (age, fare, class) have low signal for a classifier. Feature
engineering extracts higher-signal representations: family size, fare-per-person, age
bands, and binary sex encoding. These features are the direct inputs to the model.

**What we demonstrate:**
- A second `Task` pipeline stage chaining from the ingestion output
- `MLflowMonitor` loaded via `load_monitors()` — the same path `ubunye run` takes
- Lineage recording that captures both the ingestion and feature engineering runs

In [ ]:
class FeatureEngineeringTask(Task):
    """
    Stage 2 — Feature Engineering.

    Transforms clean passenger data into the feature set used for model training:
    - is_female       : binary sex encoding
    - family_size     : SibSp + Parch + 1 (self)
    - is_alone        : 1 if travelling alone
    - fare_per_person : normalised fare cost
    - age_band        : ordinal age grouping (child/adult/senior)

    Also imputes missing Age values with the median.
    """

    FEATURE_COLS = ["Survived", "Pclass", "is_female", "Age",
                    "family_size", "is_alone", "fare_per_person", "age_band"]

    def setup(self) -> None:
        pass

    def transform(self, sources: Dict[str, Any]) -> Dict[str, Any]:
        df = sources["clean_passengers"].copy()

        # Impute missing Age with median
        df["Age"] = df["Age"].fillna(df["Age"].median()).astype(int)

        # Binary sex encoding
        df["is_female"] = (df["Sex"] == "female").astype(int)

        # Family presence features
        df["family_size"] = df["SibSp"] + df["Parch"] + 1
        df["is_alone"]    = (df["family_size"] == 1).astype(int)

        # Normalise fare by family size (removes group-ticket distortion)
        df["fare_per_person"] = (df["Fare"] / df["family_size"]).round(2)

        # Ordinal age band
        df["age_band"] = pd.cut(
            df["Age"],
            bins=[0, 12, 18, 60, 100],
            labels=[0, 1, 2, 3],  # child, teen, adult, senior
        ).astype(int)

        feature_df = df[self.FEATURE_COLS].copy()
        print(f"  Features engineered: {list(feature_df.columns)}")
        print(f"  Feature set shape: {feature_df.shape}")
        print(f"  Survival rate: {feature_df['Survived'].mean():.1%}")

        return {"feature_set": feature_df}

print("FeatureEngineeringTask defined.")

In [ ]:
from ubunye.telemetry.monitors import load_monitors

feat_cfg = load_config(str(feat_dir))
feat_config_dict = feat_cfg.model_dump(mode="json")

# load_monitors reads the 'monitors' section from config —
# this is exactly what `ubunye run` does internally.
feat_monitors = load_monitors(feat_config_dict)
feat_monitors.insert(0, LineageRecorder(store="filesystem", base_dir=LINEAGE_DIR))
print(f"Monitors loaded: {[type(m).__name__ for m in feat_monitors]}")

feat_ctx = MockContext(
    run_id=str(uuid.uuid4()),
    task_name="titanic/features/feat_engineering",
)

for m in feat_monitors:
    safe_call(m, "task_start", context=feat_ctx, config=feat_config_dict)

feat_task = FeatureEngineeringTask(config=feat_config_dict)
feat_task.setup()

t0 = time.perf_counter()
feat_outputs = feat_task.transform({"clean_passengers": clean_df})
duration = time.perf_counter() - t0

feature_df = feat_outputs["feature_set"]

for m in feat_monitors:
    safe_call(m, "task_end",
              context=feat_ctx,
              config=feat_config_dict,
              outputs=feat_outputs,
              status="success",
              duration_sec=duration)

print(f"\nFeature engineering complete in {duration:.3f}s  |  run_id: {feat_ctx.run_id[:8]}...")
feature_df.head(3)

---
## Section 6 — Define the Model (UbunyeModel Contract)

**Problem it solves:**  
The engine must work with models from any library (sklearn, XGBoost, PyTorch, custom).
Locking the engine to a specific ML library makes it brittle and forces version coupling.

`UbunyeModel` is the abstract contract the engine uses to interact with *any* model.
The engine never imports sklearn or any ML library — it only calls `train`, `predict`,
`save`, `load`, and `metadata`. The user's implementation owns the ML details.

**The model file is also written to disk** so `ModelLoader` and `ModelTransform` can
dynamically import it from the task directory — the same mechanism `ubunye run` uses.

In [ ]:
# Write model.py to the training task directory.
# ModelTransform and ModelLoader will dynamically import this file.

(train_dir / "model.py").write_text(textwrap.dedent("""
    import os, pickle
    from typing import Any, Dict
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    from ubunye.models.base import UbunyeModel


    class TitanicSurvivalModel(UbunyeModel):
        \"\"\"
        Predicts Titanic passenger survival using a Random Forest classifier.

        Features used: Pclass, is_female, Age, family_size,
                       is_alone, fare_per_person, age_band
        \"\"\"

        FEATURES = ["Pclass", "is_female", "Age",
                    "family_size", "is_alone", "fare_per_person", "age_band"]
        TARGET   = "Survived"

        def __init__(self, n_estimators: int = 200, max_depth: int = 6):
            self.clf         = None
            self.n_estimators = n_estimators
            self.max_depth   = max_depth

        def train(self, df: pd.DataFrame) -> Dict[str, Any]:
            X = df[self.FEATURES]
            y = df[self.TARGET]
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y)

            self.clf = RandomForestClassifier(
                n_estimators=self.n_estimators,
                max_depth=self.max_depth,
                random_state=42,
            )
            self.clf.fit(X_train, y_train)

            y_pred = self.clf.predict(X_test)
            y_prob = self.clf.predict_proba(X_test)[:, 1]

            return {
                "accuracy": round(accuracy_score(y_test, y_pred), 4),
                "f1":       round(f1_score(y_test, y_pred), 4),
                "auc":      round(roc_auc_score(y_test, y_prob), 4),
            }

        def predict(self, df: pd.DataFrame) -> pd.DataFrame:
            out = df.copy()
            X   = df[[c for c in self.FEATURES if c in df.columns]].fillna(0)
            out["predicted_survival"]    = self.clf.predict(X)
            out["survival_probability"]  = self.clf.predict_proba(X)[:, 1].round(3)
            return out

        def save(self, path: str) -> None:
            os.makedirs(path, exist_ok=True)
            with open(os.path.join(path, "model.pkl"), "wb") as f:
                pickle.dump(self.clf, f)

        @classmethod
        def load(cls, path: str) -> "TitanicSurvivalModel":
            obj = cls()
            with open(os.path.join(path, "model.pkl"), "rb") as f:
                obj.clf = pickle.load(f)
            return obj

        def metadata(self) -> Dict[str, Any]:
            import sklearn
            return {
                "library":         "sklearn",
                "library_version": sklearn.__version__,
                "features":        self.FEATURES,
                "params": {
                    "n_estimators": self.n_estimators,
                    "max_depth":    self.max_depth,
                },
            }
"""))

print(f"model.py written to: {train_dir / 'model.py'}")

---
## Section 7 — Model Training via ModelTransform

**Problem it solves:**  
Training a model in an ad-hoc script means the trained artifact lives on someone's laptop,
metrics are in a notebook cell that got re-run, and no one knows which code produced which
model. Reproducibility is zero.

`ModelTransform` is the `type: model` plugin. When `action: train`, it:
1. Dynamically imports your `UbunyeModel` subclass from the task directory
2. Calls `model.train(df)` and captures metrics
3. Registers the artifact in `ModelRegistry` (versioned, filesystem-backed)
4. Optionally promotes to staging if `PromotionGate` thresholds pass

Every trained model gets a version number, timestamps, metrics, and lineage run ID.

In [ ]:
from ubunye.plugins.transforms.model_transform import ModelTransform

# MockBackend: the engine normally provides this; we supply a minimal stand-in.
# ModelTransform only calls getattr(backend, 'run_id', None) on it.
class MockBackend:
    run_id = ingest_ctx.run_id  # link training to the ingestion lineage run

train_transform = ModelTransform()

# The cfg dict is the 'params' sub-dict from config.yaml's transform section
train_params = {
    "action":      "train",
    "model_class": "model.TitanicSurvivalModel",
    "model_dir":   str(train_dir),      # directory containing model.py
    "registry": {
        "store":     MODEL_STORE,
        "use_case":  "titanic",
        "auto_version": True,           # generate 1.0.0 automatically
        "promote_to": "staging",        # auto-promote if gates pass
        "promotion_gates": {
            "min_accuracy": 0.72,
            "min_f1":       0.68,
        },
    },
}

print("Training TitanicSurvivalModel v1 ...")
result = train_transform.apply(
    inputs={"feature_set": feature_df},
    cfg=train_params,
    backend=MockBackend(),
)

metrics_v1 = result["model_metrics"]
print(f"\nTraining metrics (v1):")
for k, v in metrics_v1.items():
    print(f"  {k:<12}: {v}")

---
## Section 8 — Promotion Gates

**Problem it solves:**  
Without quality gates, a poorly-trained model can be promoted to production accidentally.
Manual reviews are slow and inconsistent. Promotion gates automate the quality check:
a model cannot advance to staging or production unless it meets every configured threshold.

Gates support `min_<metric>`, `max_<metric>`, and `require_drift_check`.
Failed gates return a descriptive error showing exactly which thresholds were missed.

In [ ]:
from ubunye.models.gates import PromotionGate

gate = PromotionGate({
    "min_accuracy": 0.72,
    "min_f1":       0.68,
    "min_auc":      0.80,
    "max_loss":     0.30,   # loss not in our metrics — will fail gracefully
})

print("Gate evaluation against v1 metrics:")
print(f"  Metrics: {metrics_v1}\n")

for result in gate.evaluate(metrics_v1):
    status = "✓ PASS" if result.passed else "✗ FAIL"
    print(f"  [{status}] {result.gate_name}: {result.message}")

print(f"\n  All passed: {gate.all_passed(metrics_v1)}")
failed = gate.failed_gates(metrics_v1)
if failed:
    print(f"  Failed gates: {[g.gate_name for g in failed]}")

In [ ]:
# Demonstrate gate blocking: try to promote with an impossibly high threshold.
from ubunye.models.registry import ModelRegistry, ModelStage

registry = ModelRegistry(MODEL_STORE)
versions = registry.list_versions("titanic", "TitanicSurvivalModel")
v1_version = versions[0].version
print(f"Registered version: {v1_version}  stage: {versions[0].stage.value}")

# Try to promote directly to production with a threshold the model cannot meet
try:
    registry.promote(
        use_case="titanic",
        model_name="TitanicSurvivalModel",
        version=v1_version,
        to_stage=ModelStage.PRODUCTION,
        gates={"min_accuracy": 0.99},   # impossible threshold
    )
except ValueError as e:
    print(f"\nPromotion blocked (expected):\n{e}")

---
## Section 9 — Model Registry & Promotion Lifecycle

**Problem it solves:**  
Without a registry, teams have no visibility into which model is in production,
what its performance was, who promoted it, or how to roll back if something goes wrong.

The `ModelRegistry` provides a full lifecycle: `development → staging → production → archived`.
All transitions are timestamped and auditable. Only one version can be in `production` at
a time — promoting a new version automatically archives the previous one.

**What we demonstrate:**
- Inspecting the registry via Python API and CLI
- Promoting staging → production
- Every CLI model sub-command (`list`, `info`, `promote`, `demote`)

In [ ]:
# List all versions — v1 should be in 'staging' (auto-promoted by ModelTransform)
print("Current registry state:")
for mv in registry.list_versions("titanic", "TitanicSurvivalModel"):
    print(f"  v{mv.version:<8} stage={mv.stage.value:<12} "
          f"registered={mv.registered_at[:19]}  metrics={mv.metrics}")

# Promote v1 from staging → production
prod_mv = registry.promote(
    use_case="titanic",
    model_name="TitanicSurvivalModel",
    version=v1_version,
    to_stage=ModelStage.PRODUCTION,
    promoted_by="data-scientist@company.com",
    gates={"min_accuracy": 0.70},   # realistic threshold
)
print(f"\nPromoted v{prod_mv.version} to {prod_mv.stage.value}")
print(f"  promoted_by: {prod_mv.promoted_by}")
print(f"  promoted_at: {prod_mv.promoted_to_prod[:19]}")

In [ ]:
# ── ubunye models CLI commands ────────────────────────────────────────────
MODEL_NAME = "TitanicSurvivalModel"

# list — tabular view of all versions
cli("models", "list",
    "-u", "titanic", "-m", MODEL_NAME, "-s", MODEL_STORE)

# info — full JSON for v1
cli("models", "info",
    "-u", "titanic", "-m", MODEL_NAME, "-s", MODEL_STORE, "-v", v1_version)

---
## Section 10 — Production Prediction

**Problem it solves:**  
Serving a model for batch prediction requires knowing which version is in production,
loading its artifact, and applying it to new data. Without a registry this is done by
passing file paths around in tickets and Slack messages.

`ModelTransform` with `action: predict` loads the production version from the registry
automatically — no path management, no manual version tracking. Change the production
version in the registry and the next prediction run picks it up with zero code changes.

In [ ]:
# Load the Titanic test set (no labels — simulates live scoring)
test_raw = pd.read_csv("/kaggle/input/titanic/test.csv")

# Apply the same feature engineering as training
# (in production this would be a shared feature pipeline run)
test_df = test_raw.copy()
test_df["Age"]           = test_df["Age"].fillna(test_df["Age"].median()).astype(int)
test_df["is_female"]     = (test_df["Sex"] == "female").astype(int)
test_df["family_size"]   = test_df["SibSp"] + test_df["Parch"] + 1
test_df["is_alone"]      = (test_df["family_size"] == 1).astype(int)
test_df["fare_per_person"] = (test_df["Fare"].fillna(test_df["Fare"].median()) / test_df["family_size"]).round(2)
test_df["age_band"]      = pd.cut(test_df["Age"],
                                   bins=[0,12,18,60,100],
                                   labels=[0,1,2,3]).astype(int)

# ModelTransform with action: predict — loads the production model from registry
predict_params = {
    "action":      "predict",
    "model_class": "model.TitanicSurvivalModel",
    "model_dir":   str(train_dir),
    "registry": {
        "store":     MODEL_STORE,
        "use_case":  "titanic",
        "use_stage": "production",   # always load from production
    },
}

predict_result = train_transform.apply(
    inputs={"test_data": test_df},
    cfg=predict_params,
    backend=MockBackend(),
)

predictions_df = predict_result["predictions"]
print(f"Predictions generated for {len(predictions_df)} passengers")
print(f"Predicted survivors: {predictions_df['predicted_survival'].sum()} "
      f"({predictions_df['predicted_survival'].mean():.1%})")
print()
predictions_df[["PassengerId", "Pclass", "is_female", "Age",
                "predicted_survival", "survival_probability"]].head(10)

In [ ]:
# Alternative: load model directly from registry Python API (no ModelTransform)
# Useful when you want the model instance for inspection or custom scoring.
from ubunye.models.loader import load_model_class

artifact_path, prod_version = registry.get_model(
    use_case="titanic",
    model_name="TitanicSurvivalModel",
    stage=ModelStage.PRODUCTION,
)

ModelClass = load_model_class(str(train_dir), "model.TitanicSurvivalModel")
prod_model  = ModelClass.load(artifact_path)

print(f"Loaded: {prod_version.version}  stage: {prod_version.stage.value}")
print(f"Artifact path: {artifact_path}")
print(f"Model metadata: {prod_model.metadata()}")

# Feature importances — useful for model explanation
importances = pd.Series(
    prod_model.clf.feature_importances_,
    index=prod_model.FEATURES
).sort_values(ascending=False)
print("\nFeature importances:")
print(importances.to_string())

---
## Section 11 — Lineage Inspection

**Problem it solves:**  
When a data anomaly appears in production, the team needs to answer:
- *What data was used to produce this output?*
- *Did the schema change between yesterday's run and today's?*
- *Which run produced the model that's now in production?*

Ubunye Engine automatically records every run as a structured JSON provenance record.
The lineage CLI lets you list runs, inspect a single run's full detail, trace the
data flow graph, compare two runs side-by-side, and search across all tasks by
status or date.

In [ ]:
# ubunye lineage list — recent runs for ingestion task
# The lineage CLI combines --usecase-dir and --lineage-dir to find records:
# full_path = PIPELINE_ROOT / ".ubunye" / "lineage"
cli("lineage", "list",
    "-d", str(PIPELINE_ROOT),
    "-u", "titanic", "-p", "ingestion", "-t", "raw_ingest",
    "--lineage-dir", ".ubunye/lineage",
    "-n", "5")

In [ ]:
# ubunye lineage show — full JSON of the latest ingestion run
cli("lineage", "show",
    "-d", str(PIPELINE_ROOT),
    "-u", "titanic", "-p", "ingestion", "-t", "raw_ingest",
    "--lineage-dir", ".ubunye/lineage")

In [ ]:
# ubunye lineage trace — data flow graph: inputs → transform → outputs
# Shows format, location, row count, schema hash, and data hash for each step.
cli("lineage", "trace",
    "-d", str(PIPELINE_ROOT),
    "-u", "titanic", "-p", "features", "-t", "feat_engineering",
    "--lineage-dir", ".ubunye/lineage")

In [ ]:
# ubunye lineage search — find all successful runs across all tasks
cli("lineage", "search",
    "-d", str(PIPELINE_ROOT),
    "--lineage-dir", ".ubunye/lineage",
    "--status", "success")

In [ ]:
# ubunye lineage compare — diff two runs (need 2 runs of the same task)
# Run ingestion a second time to get a second lineage record to compare

ingest_ctx2 = MockContext(
    run_id=str(uuid.uuid4()),
    task_name="titanic/ingestion/raw_ingest",
)
lineage_r2 = LineageRecorder(store="filesystem", base_dir=LINEAGE_DIR)
safe_call(lineage_r2, "task_start", context=ingest_ctx2, config=ingest_config_dict)
# Simulate a slightly different result (e.g., drop one extra row)
outputs2 = {"clean_passengers": clean_df.iloc[:-5]}
safe_call(lineage_r2, "task_end",
          context=ingest_ctx2, config=ingest_config_dict,
          outputs=outputs2, status="success", duration_sec=0.9)

# Now compare the two runs
cli("lineage", "compare",
    "-d", str(PIPELINE_ROOT),
    "-u", "titanic", "-p", "ingestion", "-t", "raw_ingest",
    "--lineage-dir", ".ubunye/lineage",
    "--run-id1", ingest_ctx.run_id,
    "--run-id2", ingest_ctx2.run_id)

---
## Section 12 — Model Maintenance

**Problem it solves:**  
Models degrade over time as data distributions shift. The team needs to:
- Retrain a new version with updated data or tuned hyperparameters
- Compare the new version against the current production model before promoting
- Roll back instantly if a newly promoted model performs worse in production
- Archive old versions that are no longer needed

**What we demonstrate:**
- Training v2 with different hyperparameters
- Side-by-side metric comparison (Python API + CLI)
- Emergency rollback to v1
- Archiving a stale version

In [ ]:
# Train v2 — deeper trees, more estimators (simulates a retraining run)
train_params_v2 = {
    "action":      "train",
    "model_class": "model.TitanicSurvivalModel",
    "model_dir":   str(train_dir),
    "registry": {
        "store":     MODEL_STORE,
        "use_case":  "titanic",
        "auto_version": True,           # auto-increments to 1.0.1
        "promote_to": None,             # register only; don't auto-promote
    },
}

# Override hyperparameters by patching model.py temporarily
# (in production you'd pass these via config or environment)
original_model_src = (train_dir / "model.py").read_text()
v2_model_src = original_model_src.replace(
    "n_estimators: int = 200, max_depth: int = 6",
    "n_estimators: int = 300, max_depth: int = 8"
)
(train_dir / "model.py").write_text(v2_model_src)

print("Training TitanicSurvivalModel v2 (n_estimators=300, max_depth=8)...")
result_v2 = train_transform.apply(
    inputs={"feature_set": feature_df},
    cfg=train_params_v2,
    backend=MockBackend(),
)
metrics_v2 = result_v2["model_metrics"]
print(f"\nv2 metrics: {metrics_v2}")

# Restore original model.py
(train_dir / "model.py").write_text(original_model_src)

In [ ]:
# Compare v1 vs v2 using the Python API
versions = registry.list_versions("titanic", "TitanicSurvivalModel")
v2_version = next(v.version for v in versions if v.version != v1_version)

print(f"Comparing v{v1_version} (production) vs v{v2_version} (development):")
diff = registry.compare_versions("titanic", "TitanicSurvivalModel", v1_version, v2_version)
for metric, vals in diff.items():
    delta_str = f"{vals['delta']:+.4f}" if vals["delta"] is not None else "n/a"
    direction = "↑" if vals["delta"] and vals["delta"] > 0 else "↓" if vals["delta"] and vals["delta"] < 0 else "="
    print(f"  {metric:<12} v{v1_version}={vals['a']}  v{v2_version}={vals['b']}  delta={delta_str} {direction}")

# CLI version of the same comparison
print()
cli("models", "compare",
    "-u", "titanic", "-m", MODEL_NAME, "-s", MODEL_STORE,
    "--versions", v1_version, "--versions", v2_version)

In [ ]:
# Promote v2 to production (v1 is automatically archived)
registry.promote("titanic", "TitanicSurvivalModel", v2_version, ModelStage.STAGING,
                 promoted_by="data-scientist@company.com")
registry.promote("titanic", "TitanicSurvivalModel", v2_version, ModelStage.PRODUCTION,
                 promoted_by="data-scientist@company.com",
                 gates={"min_accuracy": 0.70})

print("After promoting v2 to production:")
for mv in registry.list_versions("titanic", "TitanicSurvivalModel"):
    print(f"  v{mv.version:<6} {mv.stage.value}")

In [ ]:
# ROLLBACK: v2 has a production issue — roll back to v1 immediately.
# rollback() archives v2 and restores v1 to production in a single atomic operation.

rolled_back = registry.rollback(
    use_case="titanic",
    model_name="TitanicSurvivalModel",
    to_version=v1_version,
)
print(f"Rolled back — production is now: v{rolled_back.version}  stage={rolled_back.stage.value}")
print()

# CLI rollback equivalent
# cli("models", "rollback", "-u", "titanic", "-m", MODEL_NAME, "-s", MODEL_STORE, "-v", v1_version)

print("Registry state after rollback:")
for mv in registry.list_versions("titanic", "TitanicSurvivalModel"):
    print(f"  v{mv.version:<6} {mv.stage.value}")

In [ ]:
# Archive v2 — keeps the artifact for audit purposes but marks it as retired.
cli("models", "archive",
    "-u", "titanic", "-m", MODEL_NAME, "-s", MODEL_STORE, "-v", v2_version)

# Final registry state
cli("models", "list",
    "-u", "titanic", "-m", MODEL_NAME, "-s", MODEL_STORE)

---
## Section 13 — Inspect MLflow Experiments

**Problem it solves:**  
After many training runs, teams lose track of which experiments were run with which
parameters and what the outcomes were. MLflow provides a structured experiment tracker
that stores all params, metrics, and tags — queryable by experiment or run.

Here we inspect everything that was automatically logged by `MLflowMonitor` during
the ingestion and feature engineering runs.

In [ ]:
client = mlflow.tracking.MlflowClient()

for exp_name in ["titanic_ingestion", "titanic_feature_engineering"]:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"{exp_name}: no runs yet")
        continue

    runs = client.search_runs(experiment_ids=[exp.experiment_id])
    print(f"\nExperiment: {exp_name}  ({len(runs)} run(s))")
    print("-" * 60)
    for run in runs:
        print(f"  Run  : {run.info.run_name}  [{run.info.run_id[:8]}]")
        print(f"  Status : {run.info.status}")
        if run.data.metrics:
            print(f"  Metrics: {run.data.metrics}")
        if run.data.tags:
            user_tags = {k: v for k, v in run.data.tags.items() if not k.startswith("mlflow.")}
            if user_tags:
                print(f"  Tags   : {user_tags}")
        print()

---
## Section 14 — CLI Reference

Quick reference of every CLI command used in this notebook.
Run `ubunye --help` or `ubunye <command> --help` for full option docs.

In [ ]:
# Show all top-level commands
cli("--help")

In [ ]:
# Show all discovered reader/writer/transform plugins
cli("plugins")

In [ ]:
# Show engine version
cli("version")

In [ ]:
# Show all models sub-commands
cli("models", "--help")

In [ ]:
# Show all lineage sub-commands
cli("lineage", "--help")

---
## Summary

This notebook walked through the complete Ubunye Engine ML lifecycle using the
Titanic survival prediction use case:

| Stage | Engine Feature | Key Class / Command |
|---|---|---|
| Project scaffold | Task folder creation | `ubunye init` |
| Config templating | Jinja2 + env vars + profiles | `load_config()`, `{{ dt }}`, `{{ env.VAR }}` |
| Validation | Schema + template check | `ubunye validate`, `ubunye plan` |
| Ingestion | User transform contract | `Task.transform()` |
| Feature engineering | Multi-stage pipeline | `Task` + `load_monitors()` |
| Telemetry | Experiment tracking | `MLflowMonitor`, `LineageRecorder` |
| Model training | Library-agnostic ML | `UbunyeModel`, `ModelTransform(action=train)` |
| Quality gates | Automated promotion checks | `PromotionGate` |
| Registry | Versioned model store | `ModelRegistry.register()`, `promote()` |
| Prediction | Registry-driven scoring | `ModelTransform(action=predict)` |
| Lineage | Full audit trail | `ubunye lineage list/show/trace/compare/search` |
| Maintenance | Rollback + comparison | `rollback()`, `compare_versions()`, `archive()` |

### Key Design Principles Demonstrated

1. **Library independence** — the engine never imports sklearn; it only calls the `UbunyeModel` contract
2. **Config-first** — all behaviour is declared in YAML; the engine interprets, not the user's code
3. **Zero engine changes** — `ModelTransform` plugs into the existing Transform dispatch via entry points
4. **Fail-safe monitors** — `safe_call()` wraps all monitor calls so telemetry failures never crash tasks
5. **One production version** — promoting a new model to production automatically archives the previous one